# Regression vs ComBat-Based Harmonisation

In this section, we compare regression-based harmonisation with ComBat-based harmonisation using simulated cross-sectional multi-site imaging data.

---

## Objectives

We will:

1. Simulate cross-sectional imaging data across multiple sites.
2. Introduce known:
   - additive batch effects (mean shifts),
   - multiplicative batch effects (variance differences).
3. Apply regression-based harmonisation.
4. Visualize the data before and after regression correction.
5. Apply ComBat/neuroHarmonize harmonisation.
6. Compare the harmonized results using visualizations and statistical evaluation.

---

## Why this matters

Scanner and site effects can introduce unwanted variability into imaging measurements.

These effects may appear as:
- systematic shifts in mean values across scanners,
- or differences in measurement variability across sites.

Simple regression approaches can often reduce additive mean shifts, but may not fully address multiplicative variance effects.

ComBat-like harmonisation methods are specifically designed to model and correct both:
- additive effects,
- and multiplicative effects,

while preserving biological variability associated with covariates of interest.

---

## Simulated Batch Effects

For demonstration purposes, the simulated batch effects in this example are intentionally exaggerated to make scanner/site-related effects easier to visualize and evaluate.

The simulated features include:

| Feature | Simulated effect |
|---|---|
| Region_1 | additive effect only |
| Region_2 | multiplicative effect only |
| Region_3 | additive + multiplicative effects |
| Region_4 | minimal/no batch effect |

---

## Expected observations

### After regression harmonisation
- additive site shifts should reduce,
- but variance differences may still remain.

### After ComBat/neuroHarmonize harmonisation
- both mean shifts and variance differences should reduce more effectively.

---

In [ ]:
# If needed
# !pip install neuroHarmonize
# !pip install nibabel
# !pip install neuroHarmonize neuroCombat

In [25]:
# Load packages
import importlib
import HarmonisationEvaluation_functions
import HarmonisationEvaluation_plots
import pandas as pd

importlib.reload(HarmonisationEvaluation_functions)
importlib.reload(HarmonisationEvaluation_plots)

from HarmonisationEvaluation_functions import simulate_cross_sectional_harmonization_data
from HarmonisationEvaluation_plots import plot_additive_multiplicative_effects
from HarmonisationEvaluation_plots import plot_before_after_by_site
from neuroHarmonize import harmonizationLearn

In [ ]:
# Generate simulated data

# Define known site effects
# -----------------------------
# Simulated batch effects
# -----------------------------

# Region_1 -> additive only
# Region_2 -> multiplicative only
# Region_3 -> additive + multiplicative
# Region_4 -> no batch effect

additive_shift = {
    "Site_B": {
        "Region_1": 300,
        "Region_3": -250,
    },

    "Site_C": {
        "Region_1": -200,
        "Region_3": 220,
    },
}

multiplicative_scale = {
    "Site_A": {
        "Region_2": 7.0,
        "Region_3": 3.0,
    },

    "Site_B": {
        "Region_2": 6.8,
        "Region_3": 3.5,
    },

    "Site_C": {
        "Region_2": 0.3,
        "Region_3": 5.5,
    },
}

df = simulate_cross_sectional_harmonization_data(
    n_subjects=300,
    n_sites=3,
    n_features=4,
    seed=1,
    additive_shift=additive_shift,
    multiplicative_scale=multiplicative_scale,
)
print(df.head())

feature_cols = ["Region_1", "Region_2", "Region_3", "Region_4"]
plot_additive_multiplicative_effects(df, feature_cols=feature_cols, batch_col="Site")


In [ ]:
from HarmonisationEvaluation_functions import regression_harmonize_site

feature_cols = ["Region_1", "Region_2", "Region_3", "Region_4"]

df_harm, model_summary = regression_harmonize_site(
    df,
    feature_cols=feature_cols,
    site_col="Site",
    covariates=("Age", "Timepoint")
)

print(model_summary)
print(df_harm.head())
plot_before_after_by_site(df_harm, feature_cols)

In [ ]:
# Harmonise using ComBat

feature_cols = ["Region_1", "Region_2", "Region_3", "Region_4"]

# original feature matrix
data = df[feature_cols].to_numpy()

# covariates dataframe
covars = pd.DataFrame({
    "SITE": df["Site"],
    "Age": df["Age"]
})

# harmonize
model, data_harm = harmonizationLearn(data, covars)

# convert harmonized matrix to dataframe
df_harm = pd.DataFrame(
    data_harm,
    columns=[f"{c}_harm" for c in feature_cols]
)

# combine with original dataframe
df_combined = pd.concat([df.reset_index(drop=True),
                         df_harm.reset_index(drop=True)],
                        axis=1)

print(df_combined)

# Plot additive and multiplicative effects after combat
plot_before_after_by_site(df_combined, feature_cols)

---
## Exercise

Modify the simulation to include:
- more imaging regions/features,
- different combinations of additive and multiplicative batch effects,
- or additional sites/scanners.

Then:

1. Apply regression-based harmonisation.
2. Apply ComBat/neuroHarmonize harmonisation.
3. Compare the results visually and statistically.

### Questions

- Which regions improved most after regression harmonisation?
- Which regions still showed residual variance differences?
- Did ComBat-based harmonisation further reduce batch effects?
- Which simulated effects were easiest or hardest to correct?

### Site regression when you have timepoints - 
Refer to the workbook - SiteRegression_forLongHarmonisation_workbook2.ipynb

---

# Longitudinal ComBat Harmonisation

Standard ComBat methods were originally developed for cross-sectional data, where each subject contributes a single observation.

In longitudinal studies, however, subjects are measured repeatedly over time.  
These repeated measurements introduce within-subject correlations that should be accounted for during harmonisation.

---

## What does longitudinal ComBat do?

Longitudinal ComBat extends the ComBat framework by incorporating subject-level random effects within a mixed-effects modeling framework.

This allows the method to:
- model repeated measurements from the same subject,
- preserve longitudinal biological trajectories,
- and estimate additive and multiplicative batch effects more appropriately for longitudinal data.

---

## Conceptual difference from standard ComBat

### Standard ComBat
Primarily models:

$$
y_{ij} = \alpha_j + \beta_j X_i + \gamma_{b(i)j} + \delta_{b(i)j}\epsilon_{ij}
$$

where:
- $\gamma$ represents additive batch effects,
- $\delta$ represents multiplicative batch effects.

This assumes observations are independent.

---

### Longitudinal ComBat
Adds subject-level random effects:

$$
y_{ij}(t) =
\alpha_j + \beta_j X_i(t)
+ u_i
+ \gamma_{b(i)j}
+ \delta_{b(i)j}\epsilon_{ij}
$$

where:
- $u_i$ represents subject-specific random effects,
- repeated measurements within subjects are explicitly modeled.

This helps preserve within-subject longitudinal structure during harmonisation.

---

## Current Implementation

At present, longitudinal ComBat is primarily available through an R implementation.

One commonly used implementation is:

- **longCombat (R package)**  
  https://github.com/jcbeer/longCombat

Python packages such as:
- `neuroCombat`
- `neuroHarmonize`

mainly implement standard (cross-sectional) ComBat approaches.

---

## Key Takeaway

For longitudinal imaging studies, explicitly modeling repeated subject measurements is important for reliable harmonisation.

Longitudinal ComBat extends standard ComBat by accounting for within-subject dependence while still correcting additive and multiplicative scanner/site effects.